In [1]:
import time
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import os


def download_lot_html_by_reg(auction_id):
    base_url = f"https://app.mag.co.uk/auctions/auction/{auction_id}"

    options = ChromeOptions()
    options.headless = False

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    try:
        driver.get(base_url)
        driver.maximize_window()
        time.sleep(3)
        try:
            title_elem = driver.find_element(By.CSS_SELECTOR, "span.navy")
            auction_title = title_elem.text.strip()
        except:
            auction_title = "Unknown Auction Title"

        print("Auction Title:", auction_title)

 
        try:
            auc_type_elem = driver.find_element(
                By.CSS_SELECTOR, 
                "span.d-inline-flex.align-items-center.gap-1.fs-13"
            )
            auction_type = auc_type_elem.text.strip()
        except:
            auction_type = "Unknown Auction Type"

        print("Auction Type:", auction_type)


        os.makedirs("database", exist_ok=True)


        with open("database/db.json", "w", encoding="utf-8") as jf:
            json.dump(
                {
                    "auction_title": auction_title,
                    "auction_type": auction_type
                },
                jf,
                indent=4,
                ensure_ascii=False
            )

        print("Saved auction info → database/db.json")

        # ---------------------------------------
        # START SCRAPING LOTS
        # ---------------------------------------
        os.makedirs("html", exist_ok=True)

        page = 1

        while True:
            print(f"\n--- Processing Page {page} ---")
            time.sleep(2)

            lot_buttons = driver.find_elements(By.CSS_SELECTOR, "button.btn-view-lot")
            print(f"Found {len(lot_buttons)} lots on page {page}.")

            main_window = driver.current_window_handle

            for idx, btn in enumerate(lot_buttons, start=1):
                stock_id = btn.get_attribute("data-stockid")
                catalogue_id = btn.get_attribute("data-catalogueid")
                catlot_id = btn.get_attribute("data-catlotid")

                lot_url = (
                    f"https://app.mag.co.uk/view-stock-report?stockID={stock_id}"
                    f"&catalogueID={catalogue_id}&catLotID={catlot_id}&isMod"
                )

                driver.execute_script("window.open('');")
                driver.switch_to.window(driver.window_handles[1])
                driver.get(lot_url)
                time.sleep(3)

                try:
                    reg_elem = driver.find_element(By.CSS_SELECTOR, "span.reg-plate")
                    reg_number = reg_elem.text.strip().replace(" ", "")
                except:
                    reg_number = f"page{page}_lot{idx}_{stock_id}"

                html_content = driver.page_source
                file_name = f"html/{reg_number}.html"

                with open(file_name, "w", encoding="utf-8") as f:
                    f.write(html_content)

                print(f"Saved HTML: {file_name}")

                driver.close()
                driver.switch_to.window(main_window)
                time.sleep(1)

            # Pagination handling
            try:
                next_button = driver.find_element(
                    By.CSS_SELECTOR,
                    "button[aria-label='Go to the next page']:not([aria-disabled='true'])"
                )
            except:
                next_button = None

            if next_button:
                print("Moving to next page...")
                next_button.click()
                page += 1
                time.sleep(3)
            else:
                print("No more pages. Scraping complete.")
                break

    finally:
        driver.quit()


# Run
download_lot_html_by_reg(219)


Auction Title: Monday 8th December 2025 - Rotherham - Monday 08 Dec 2025 @ 16:00
Auction Type: Physical Auction
Saved auction info → database/db.json

--- Processing Page 1 ---
Found 100 lots on page 1.
Saved HTML: html/RE16DZM.html
Saved HTML: html/KS15OHF.html
Saved HTML: html/HK67KTP.html
Saved HTML: html/SK72VEL.html


InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x1100d23
	0x1100d64
	0xeee52b
	0xf2c635
	0xf5b3a6
	0xf56ed1
	0xf56846
	0xebebfd
	0xebf18e
	0xebf65d
	0x1354d64
	0x135031b
	0x136cbfa
	0x111ac28
	0x1122c2d
	0xebe7ab
	0xebddf7
	0x14a11ff
	0x768b7ba9
	0x77cec3ab
	0x77cec32f


In [8]:
import os,re,json,csv
from bs4 import BeautifulSoup
from datetime import datetime

html_folder = "html"
output_file = "data.csv"

headers = [
  
    "Auction Name",
    "Auction type",
    "Title",
    "Lot",
    "Doors",
    "Seats",
    "Body Type",
    "Vendor",
    "VIN",
    "Center",
    "Start Date",
    "Start Time",
    "D.O.R",
    "Year",
    # "Manufacturer",
    # "Model",
    # "Variant",
    "Mileage",
    "Mileage Warranted",
    "CC",
    "Colour",
    "Former Keepers",
    "Transmission",
    "Fuel Type",
    "Reg",
    "V5",
    "VAT Status",
    # "Parcel Shelf",
    "Keys",
    # "Vendor",
    "MOT Expiry Date",
    # "CAP Clean",
    # "CAP Average",
    # "CAP Below",
    # "Date/Time",
    # "Inspection Report",
    "Grade",
    # "Features",
    # "Non Runner",
    # "Brakes",
    # "Euro Status",
    "Tyres Condition",
    "Additional Information",
    "General Condition",
    "Service History",
    # "No of Services",
    # "Last Service",
    # "Last Service Mileage",
    "Service Notes",
    "Images",
    # "Damaged_images",
    # "Damage_details",
    # "Notes",

]
    
def Details(soup):
    mainDiv = soup.find("div", class_="row vehicle-info mb-5")
    if not mainDiv:
        return {}
     
    details = {}


    for ul_tag in mainDiv.find_all("ul", class_="mag-ul"):

        for li in ul_tag.find_all("li"):

            strong_tag = li.find("strong")
            if not strong_tag:
                continue

            key = strong_tag.get_text(strip=True).replace(":", "")


            span_tag = li.find("span")
            if span_tag:
                value = span_tag.get_text(strip=True)
            else:
                value = strong_tag.next_sibling
                if value:
                    value = value.strip()
                else:
                    value = ""

            details[key] = value

    return details

def extract_specification(soup):
    spec_section = soup.find("div", id="stock-features")
    if not spec_section:
        return {"Specification": {}}

    ul_tag = spec_section.find("ul")
    if not ul_tag:
        return {"Specification": {}}

    spec_dict = {}
    count = 1

    for li in ul_tag.find_all("li"):
        text = li.get_text(strip=True)
        if text:
            spec_dict[f"Note {count}"] = text
            count += 1

    return {"Specification": spec_dict}


def extract_checklist(soup):
    checklist_data = {}

    all_sections = soup.find_all("div", class_="col-md-12 mb-3")
    for section in all_sections:
        # Section title (Engine, Interior, Warning Lights...)
        title_tag = section.find("h5")
        if not title_tag:
            continue
        
        section_title = title_tag.get_text(strip=True)
        checklist_data[section_title] = {}

        # List items inside section
        ul_tag = section.find("ul", class_="mag-ul")
        if not ul_tag:
            continue

        for li in ul_tag.find_all("li"):
            spans = li.find_all("span")
            if len(spans) >= 2:
                question = spans[0].get_text(strip=True)
                answer = spans[1].get_text(strip=True).upper()

                checklist_data[section_title][question] = answer

    return checklist_data

def extract_images(soup):
    base_url = "https://app.mag.co.uk"
    images = []

    thumb_div = soup.find("div", class_="thumbnails")
    if not thumb_div:
        return ""

    for a in thumb_div.find_all("a", class_="stock-image"):
        href = a.get("href")
        if href:

            full_url = base_url + href.strip()
            images.append(full_url)

    return ",".join(images)

def extract_auction_open_datetime(soup):

    date_div = soup.find("div", string=lambda text: text and "Auction opens" in text)
    
    if not date_div:
        return {
            "Auction_Opens_Date": None,
            "Auction_Opens_Time": None
        }
    
    text = date_div.get_text(strip=True)
 
    match = re.search(r"(\d{2}/\d{2}/\d{2})\s*@\s*(\d{2}:\d{2})", text)
    if not match:
        return {
            "Auction_Opens_Date": None,
            "Auction_Opens_Time": None
        }

    raw_date = match.group(1)   
    raw_time = match.group(2)  


    sql_date = datetime.strptime(raw_date, "%d/%m/%y").strftime("%Y-%m-%d")
    sql_time = datetime.strptime(raw_time, "%H:%M").strftime("%H:%M:%S")

    return {
        "Auction_Opens_Date": sql_date,
        "Auction_Opens_Time": sql_time
    }
    
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(headers)

    for file_name in os.listdir(html_folder):
        if not file_name.endswith(".html"):
            continue

        file_path = os.path.join(html_folder, file_name)

        with open(file_path, "r", encoding="utf-8") as f_html:
            soup = BeautifulSoup(f_html.read(), "html.parser")


            title_tag = soup.find("h2" ,class_="mb-0")
            title = title_tag.get_text(strip=True) if title_tag else ""
            
            lot_tag= soup.find("span",class_="fs-32 badge text-dark")
            lot_text = lot_tag.get_text(strip=True)
            lot = lot_text.replace("Lot","")
            
            reg_tag = soup.find("span",class_="reg-plate fs-40")
            reg_text = reg_tag.get_text(strip=True)
            get_data = Details(soup)
            raw = get_data.get("Body type","")
            Vendor = get_data.get("Vendor","")
            Colour = get_data.get("Colour","")
            Registration_date = get_data.get("Registration date","")
            Chassis_numbe = get_data.get("Chassis number","")
            Transmission_text = get_data.get("Transmission","")
            cleantext = re.sub(r"\(\d+\)","",Transmission_text)
            cleantext = cleantext.capitalize()
            Transmission = cleantext
            fuel = get_data.get("Fuel type","")

            if not raw or not isinstance(raw,str):
                door=""
                bodytype=""
            door_match = re.search(r"\((\d+)\s*doors?\)",raw,re.IGNORECASE)
            door = int(door_match.group(1)) if door_match else ""
            
            cleaned = re.sub(r"\(\d+\s*doors?\)","",raw).strip()
            cleaned = re.sub(r"^\d+\s*Door\s*","",cleaned,flags=re.IGNORECASE).strip()
            bodytype= cleaned
            
            milage_text = get_data.get("Mileage")
            num_match = re.search(r"([\d,]+)",milage_text)
            milage = int(num_match.group(1).replace(",","")) if num_match else None
            
            war_match = re.search(r"\((.*?)\)",milage_text)
            Mileage_War = war_match.group(1).strip() if war_match else ""
            
            Cc_text = get_data.get("Engine size")
            cc_text = int(Cc_text.replace("cc",""))
            cc = round(cc_text / 1000 ,1)
       
            MOT_expires = get_data.get("MOT expires","")
            centers_te = get_data.get("Location","")
            centers = centers_te.replace("MAG -" , "")
            keys= get_data.get("Keys","")
            VAT_status= get_data.get("VAT status","")
            keepers = get_data.get("Prev keepers","")
            Service_history = get_data.get("Service history","")
            Seats = get_data.get("Seats","")
            V5totle = get_data.get("V5/Logbook","")
            
            
            grade_tag = soup.find("span",class_="grade")
            grade_value = ""
            if grade_tag:
                image_tag = grade_tag.find("img")
                if image_tag:
                    grade_value = image_tag["title"].strip()
                    
                    
            tyre_div = soup.find("div",class_="tyre-depths")
            tyresData = "" 
            if tyre_div:
                depths=[]
                for div in tyre_div.find_all("div"):
                    strong = div.find("div")
                    if strong:
                        key = strong.get_text(strip=True).replace(":", "")
                        value = div.get_text(strip=True).replace(key, "").replace(":", "").strip()
                        depths.append(f"{key}:{value}")
                tyresData = ", ".join(depths)

            dateEndTime=extract_auction_open_datetime(soup)
            ServiceNotes = soup.find("div",class_="row vehicle-info mt-3 mb-5")
            ServiceNotes_text = ""
            if ServiceNotes:
                ServiceNotes_p = ServiceNotes.find("p")
                ServiceNotes_text = ServiceNotes.get_text(strip=True)
            
            with open("database/db.json","r",encoding="utf-8")as db:
                database = json.load(db)
            auction_name= "" 
            auction_type= "" 
            if database:
                auction_name = database.get("auction_title")
                auction_type = "Online Auction" if database.get("auction_type") == 'Physical Auction' else database.get("auction_type") 
                 
        data = {
            "Auction Name": auction_name,
            "Auction type": auction_type,
            "Title": title,
            "Start Date": dateEndTime.get("Auction_Opens_Date"),
            "Start Time": dateEndTime.get("Auction_Opens_Time"),
            "Lot":lot,
            "Reg":reg_text,
            "Doors":door,
            "Seats":Seats,
            "Body Type":bodytype,
            "Vendor":Vendor,
            "D.O.R":Registration_date,
            "Year":Registration_date.split("/")[-1],
            "V5":V5totle,
            "VIN":Chassis_numbe,
            "Transmission":Transmission,
            "Colour":Colour,
            "Fuel Type":fuel,
            "Mileage":milage,
            "Mileage Warranted":Mileage_War,
            "CC":cc,
            "MOT Expiry Date":MOT_expires,
            "Center":centers,
            "Keys":keys,
            "VAT Status":VAT_status,
            "Grade":grade_value,
            "Former Keepers":keepers,
            "Tyres Condition":tyresData,
            "Service History":Service_history,
            "General Condition":extract_specification(soup),
            "Additional Information":extract_checklist(soup),
            "Service Notes":ServiceNotes_text,
            "Images":extract_images(soup),
           
        }
       

        writer.writerow([data[h] for h in headers])





In [9]:
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

email = "sultanmirza0501@icloud.com"
password = "Muhssan7865"

df = pd.read_csv("data.csv")
reg_list = df["Reg"].dropna().astype(str).tolist()

save_folder = "carcheckhtml"
os.makedirs(save_folder, exist_ok=True)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.maximize_window()

driver.get("https://totalcarcheck.co.uk/Account/Login")

try:
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.ID, "UserName"))).send_keys(email)
    wait.until(EC.presence_of_element_located((By.ID, "Password"))).send_keys(password)
    driver.find_element(By.XPATH, "//input[@type='submit' and @value='Log in']").click()
    print("✔ Logged in successfully!")
except Exception as e:
    print("❌ Login failed:", e)

time.sleep(3)


def check_reg(reg):
    """
    Returns HTML if success,
    Returns None if rate-limit error.
    """
    url = f"https://totalcarcheck.co.uk/FreeCheck?regno={reg}"
    driver.get(url)
    time.sleep(4)

    html = driver.page_source

    if "please wait 1 minute" in html.lower():
        print("⛔ Rate limit detected! Waiting 70 seconds...")
        return None 
    
    return html


for reg in reg_list:
    print(f"🔎 Checking: {reg}")

    while True:
        html = check_reg(reg)

        if html is None:
       
            time.sleep(70)
            continue


        file_path = os.path.join(save_folder, f"{reg}.html")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(html)

        print(f"📩 Saved HTML: {file_path}")
        break  

driver.quit()
print("🎉 All record HTML saved successfully!")


❌ Login failed: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x1100d23
	0x1100d64
	0xeee6dd
	0xf393a5
	0xf3977b
	0xf80382
	0xf5b534
	0xf7db13
	0xf5b2e6
	0xf2d321
	0xf2e1d4
	0x1354d64
	0x135031b
	0x136cbfa
	0x111ac28
	0x1122c2d
	0x1109028
	0x11091e9
	0x10f3578
	0x768b7ba9
	0x77cec3ab
	0x77cec32f

🔎 Checking: HK67KTP


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=143.0.7499.40)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x1100d23
	0x1100d64
	0xeee6dd
	0xecca8d
	0xf61ffb
	0xf7d44c
	0xf5b2e6
	0xf2d321
	0xf2e1d4
	0x1354d64
	0x135031b
	0x136cbfa
	0x111ac28
	0x1122c2d
	0x1109028
	0x11091e9
	0x10f3578
	0x768b7ba9
	0x77cec3ab
	0x77cec32f


In [ ]:
from bs4 import BeautifulSoup
import os, re
import pandas as pd

def findtabledetails(soup):
    data_dict = {}
    rows = soup.find_all("tr")

    for tr in rows:
        labels = tr.find_all("span", class_="cert-label")
        values = tr.find_all("span", class_="cert-data-text")

        if labels and values:
            label = labels[0].get_text(strip=True)
            value = values[0].get_text(strip=True)
            data_dict[label] = value

    return data_dict


def T_scrap_by_html_to_csv(folder="carcheckhtml", output_csv="totalcarcheck.csv"):
    if not os.path.exists(folder):
        print(f"Folder '{folder}' does not exist!")
        return

    files = sorted(os.listdir(folder))
    if not files:
        print("No HTML files found in folder.")
        return

    all_data = []

    for file in files:
        if not file.endswith(".html"):
            continue

        path = os.path.join(folder, file)
        with open(path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "html.parser")
            table_data = findtabledetails(soup)
            reg_tag = soup.find("span", id="regPlateFreeCheck")
            reg_text = reg_tag.get_text(strip=True) if reg_tag else ""
            engine_cc = table_data.get("Engine Size", "")
            engine_l = ""

            if engine_cc:
                cc_match = re.findall(r"\d+", engine_cc)
                if cc_match:
                    cc_value = int(cc_match[0])
                    engine_l = round(cc_value / 1000, 1)

            row = {
                "Reg": reg_text,
                "Make": table_data.get("Manufacturer", ""),
                "Model": table_data.get("Model", ""),
                "Variant": table_data.get("Model Detail", ""),
                "Vehicle Type": table_data.get("Vehicle Type", ""),
                "Euro Status": table_data.get("Euro Status", "")
            }

            all_data.append(row)

    df = pd.DataFrame(all_data)
    df.to_csv(output_csv, index=False, encoding="utf-8")
    print(f"🚗 Completed! Saved '{output_csv}'")



T_scrap_by_html_to_csv()


🚗 Completed! Saved 'totalcarcheck.csv'


In [ ]:
import pandas as pd

pf1 = pd.read_csv("data.csv")
pf2 = pd.read_csv("totalcarcheck.csv")

pf1["Reg"] = pf1["Reg"].astype(str).str.strip()
pf2["Reg"] = pf2["Reg"].astype(str).str.strip()

marged = pd.merge(pf1,pf2,on="Reg",how="left")
marged.to_csv("Mag_scrape.csv",index=False, encoding="utf-8")

In [ ]:
from urllib.parse import urlparse, urljoin
import threading, requests, os, re
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

df = pd.read_csv("Mag_scrape.csv")


reg_img = df[['Reg', "Images"]]

def add_watermark_to_image(image_path, text="Sourced from MAG"):
    try:
        image = Image.open(image_path).convert("RGBA")
        txt_layer = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(txt_layer)

        try:
            font = ImageFont.truetype("arial.ttf", 20)
        except:
            font = ImageFont.load_default()

        margin = 10
        bbox = draw.textbbox((0, 0), text, font=font)
        tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
        x, y = image.width - tw - margin, image.height - th - margin

        draw.rectangle([x - margin, y - margin, x + tw + margin, y + th + margin],
                       fill=(0,0,0,160))

        draw.text((x, y), text, font=font, fill=(255,255,255,200))

        watermarked = Image.alpha_composite(image, txt_layer).convert("RGB")
        watermarked.save(image_path)

        print(f"✔ Watermarked: {image_path}")

    except Exception as e:
        print(f"⚠ Watermark Error: {e}")

def download_images(data, main_folder="Images"):
    os.makedirs(main_folder, exist_ok=True)

    for index, row in data.iterrows():
        reg_no = str(row["Reg"]).strip()

    
        img_urls = [u for u in re.split(r',\s*', str(row["Images"])) if u]

        reg_folder = os.path.join(main_folder, reg_no)
        original_folder = os.path.join(reg_folder, "Original")

        os.makedirs(original_folder, exist_ok=True)

        def save_img(url, folder, idx):
            url = url.strip()
            if not url:
                return

            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            parsed = urlparse(url)
            if not parsed.netloc:
                print(f"❌ Invalid URL Skipped: {url}")
                return

            full_path = os.path.join(folder, f"{reg_no}_{idx}.jpg")

            if os.path.exists(full_path):
                print(f"⏩ Skipped (Exists): {full_path}")
                return

            try:
                response = requests.get(url, stream=True, timeout=20)
                response.raise_for_status()

                with open(full_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(full_path)
                print(f"📌 Saved: {full_path}")

            except Exception as e:
                print(f"⚠ Error downloading: {url} -> {e}")

        # save only normal images now
        for i, url in enumerate(img_urls):
            save_img(url, original_folder, i+1)

def start_funcs():
    t1 = threading.Thread(target=download_images, args=(reg_img,))
    t1.start()
    t1.join()

if __name__ == "__main__":
    start_funcs()


✔ Watermarked: Images\AE18RYB\Original\AE18RYB_1.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_1.jpg
✔ Watermarked: Images\AE18RYB\Original\AE18RYB_2.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_2.jpg
✔ Watermarked: Images\AE18RYB\Original\AE18RYB_3.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_3.jpg
✔ Watermarked: Images\AE18RYB\Original\AE18RYB_4.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_4.jpg
✔ Watermarked: Images\AE18RYB\Original\AE18RYB_5.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_5.jpg
✔ Watermarked: Images\AE18RYB\Original\AE18RYB_6.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_6.jpg
✔ Watermarked: Images\AE18RYB\Original\AE18RYB_7.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_7.jpg
✔ Watermarked: Images\AE18RYB\Original\AE18RYB_8.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_8.jpg
✔ Watermarked: Images\AE18RYB\Original\AE18RYB_9.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_9.jpg
✔ Watermarked: Images\AE18RYB\Original\AE18RYB_10.jpg
📌 Saved: Images\AE18RYB\Original\AE18RYB_10.jp